In [192]:
# Set the PySpark environment variables
import os
os.environ['SPARK_HOME'] = r"C:\Users\PMLS\Documents\apps\Spark"
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter'
os.environ['PYSPARK_DRIVER_PYTHON_OPTS'] = 'lab'
os.environ['PYSPARK_PYTHON'] = 'python'

In [193]:
from pyspark.sql import SparkSession


In [194]:
# Create a Spark session
spark = SparkSession.builder.appName("VoltmartDataCleaning").getOrCreate()

In [195]:
# Load the CSV file into a DataFrame
file_path = 'G:/FELLOWSHIP/Data-Engineering-BWF/Task10/data.csv'
df = spark.read.option("header", "true") \
               .option("quote", "\"") \
               .option("escape", "\"") \
               .option("delimiter", ",") \
               .csv(file_path)


In [196]:
# Show the first few rows and the schema of the DataFrame
df.show(5, truncate=False)
df.printSchema()

+-----------------------+--------+------------------------+-------------+------------+---------------------------------------+----------------+----------+----------+--------+-------+
|order_date             |order_id|product                 |product_id   |category    |purchase_address                       |quantity_ordered|price_each|cost_price|turnover|margin |
+-----------------------+--------+------------------------+-------------+------------+---------------------------------------+----------------+----------+----------+--------+-------+
|2023-01-22T21:25:00.000|141234  |iPhone                  |5638008983335|Vêtements   |"944 Walnut St, Boston, MA 02215"      |1               |700       |231       |700     |469    |
|2023-01-28T14:15:00.000|141235  |Lightning Charging Cable|5563319511488|Alimentation|"185 Maple St, Portland, OR 97035"     |1               |14.95     |7.475     |14.95   |7.475  |
|2023-01-17T13:33:00.000|141236  |Wired Headphones        |2113973395220|Vêtements   

In [210]:
from pyspark.sql.functions import col, hour, to_date, when, lower, split, regexp_replace

In [211]:
# Extract the hour from the 'order_date' column
df = df.withColumn("order_hour", hour(col("order_date")))

# Filter out orders between midnight and 5 AM
filtered_df = df.filter(~col("order_hour").between(0, 4))


In [212]:
from pyspark.sql import functions as F

# Create the 'time_of_day' column based on the hour using a single statement
filtered_df = filtered_df.withColumn(
    "time_of_day",
    F.when(F.col("order_hour").between(5, 11), "morning")
    .when(F.col("order_hour").between(12, 17), "afternoon")
    .when(F.col("order_hour").between(18, 23), "evening")
    .otherwise("unknown")  # Handle any values that don't fall into the defined ranges
)

In [213]:
# Convert 'order_date' to just the date
filtered_df = filtered_df.withColumn("order_date", to_date(col("order_date")))

In [214]:
filtered_df= filtered_df.drop('order_hour')

In [215]:
# Filter out rows containing "TV" (case-insensitive)
filtered_df = filtered_df.filter(lower(col("product")).contains("tv") == False)

In [216]:
# Convert "category" column to lowercase
filtered_df = filtered_df.withColumn("category", lower(col("category")))

In [223]:
# Split the purchase_address column by space
filtered_df = filtered_df.withColumn("address_parts", split(col("purchase_address"), ","))

# Extract the state (assuming it's the second-to-last part)
filtered_df = filtered_df.withColumn("purchase_state", col("address_parts")[2])

# Drop the temporary 'address_parts' column
filtered_df = filtered_df.drop("address_parts")

# Remove zipcode from the 'purchase_state' column, keeping only letters
filtered_df = filtered_df.withColumn(
    "purchase_state",
    regexp_replace(col("purchase_state"), "[^A-Za-z]", "")
)
filtered_df = filtered_df.withColumn('purchase_address', regexp_replace('purchase_address', '"', ''))

In [224]:
filtered_df.show(5, truncate=False)
filtered_df.printSchema()

+----------+--------+------------------------+-------------+------------+-------------------------------------+----------------+----------+----------+--------+-------+-----------+--------------+
|order_date|order_id|product                 |product_id   |category    |purchase_address                     |quantity_ordered|price_each|cost_price|turnover|margin |time_of_day|purchase_state|
+----------+--------+------------------------+-------------+------------+-------------------------------------+----------------+----------+----------+--------+-------+-----------+--------------+
|2023-01-22|141234  |iPhone                  |5638008983335|vêtements   |944 Walnut St, Boston, MA 02215      |1               |700       |231       |700     |469    |evening    |MA            |
|2023-01-28|141235  |Lightning Charging Cable|5563319511488|alimentation|185 Maple St, Portland, OR 97035     |1               |14.95     |7.475     |14.95   |7.475  |afternoon  |OR            |
|2023-01-17|141236  |Wire

In [231]:
parquet_output_path = ".\Task10\filtered_data.parquet"
# Save the DataFrame as a CSV file
filtered_df.write.parquet(output_path)

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\PMLS\AppData\Local\Temp\ipykernel_28780\1921297872.py:1: SyntaxWarning: invalid escape sequence '\T'
  parquet_output_path = ".\Task10\filtered_data.parquet"
C:\Users\PMLS\AppData\Local\Temp\ipykernel_28780\1921297872.py:1: SyntaxWarning: invalid escape sequence '\T'
  parquet_output_path = ".\Task10\filtered_data.parquet"


AnalysisException: [PATH_ALREADY_EXISTS] Path file:/G:/FELLOWSHIP/Data-Engineering-BWF/Task10/filtered_data already exists. Set mode as "overwrite" to overwrite the existing path.